# RQ2: Attribution Faithfulness Benchmark (Comprehensiveness, Sufficiency, LeRF & AOPC)
Full-scale evaluation of decision faithfulness via progressive Top-$k$ WordUnit removal (`Comprehensiveness`), retention (`Sufficiency`), Least-Important (`LeRF` / `near_zero`) sanity check, `Random Baseline`, Wilcoxon/Holm statistical tests, and cross-operator robustness (`deletion` vs `embedding_mask` evaluated simultaneously in a single pass) on SST-2.

Default hyperparameters match the completed RQ1 `Qwen2.5-7B` run (`N = 310`, `seed = 42`):
- **Occlusion (Leave-One-Out)**: deterministic
- **Gradient $\times$ Input**: deterministic
- **Integrated Gradients**: $m = 100$ Riemann steps (`ig_baseline = zero` for 1:1 consistency with RQ1 `Qwen2.5-7B`; use `mean` for fresh models like `Qwen2.5-3B` or `Qwen3-8B-Base`)
- **LIME**: $N = 1000$ neighborhood coalitions (with mask deduplication & dynamic batch padding)
- **SHAP (Permutation)**: $B = 2500$ evaluation budget (with in-call coalition memoization)

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False
IN_COLAB_HOSTED = IN_COLAB and os.path.isdir("/content")

# Optional: custom directory (e.g. Google Drive for Desktop on Windows: r"G:\My Drive\llm_xai_results")
LOCAL_DRIVE_RESULTS = "" #@param {type:"string"}

if IN_COLAB_HOSTED:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

    GITHUB_TOKEN = "" #@param {type:"string"}
    REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
    BRANCH = "main" #@param {type:"string"}

    try:
        from google.colab import userdata
        token_from_secrets = userdata.get('GH_TOKEN')
        if token_from_secrets:
            GITHUB_TOKEN = token_from_secrets
    except Exception:
        pass

    if GITHUB_TOKEN.strip():
        CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
    else:
        CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

    REPO_DIR = "/content/llm-xai"
    if not os.path.exists(REPO_DIR):
        res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
        print("\n".join(res))
        if not os.path.exists(REPO_DIR):
            raise RuntimeError("Cloning error! Provide Personal Access Token (PAT) in GITHUB_TOKEN or set GH_TOKEN secret.")
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    !pip install -q "transformers>=4.51.0,<5.0.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.45.1" scikit-learn scipy pandas matplotlib seaborn huggingface_hub

    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
else:
    if LOCAL_DRIVE_RESULTS.strip():
        DRIVE_RESULTS = LOCAL_DRIVE_RESULTS.strip()
    else:
        drive_candidates = [
            r"G:\My Drive",
            r"G:\Mój dysk",
            r"D:\My Drive",
            r"D:\Mój dysk",
            os.path.expanduser("~/Google Drive/My Drive"),
            os.path.expanduser("~/Google Drive/Mój dysk"),
        ]
        detected_drive = next((p for p in drive_candidates if os.path.isdir(p)), None)
        DRIVE_RESULTS = os.path.join(detected_drive, "llm_xai_results") if detected_drive else "results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    mode_lbl = "Colab Local Runtime (RTX 4090 / local GPU)" if IN_COLAB else "Local Python / Jupyter"
    print(f"Running in: {mode_lbl}. Checkpoints & results directory: {os.path.abspath(DRIVE_RESULTS)}")

### 3. Pipeline Smoke Test (2 samples, Dual-Operator Single Pass)
Quick verification that the selected Qwen model, `WordUnit` alignment, both ablation operators (`deletion` & `embedding_mask`), and all 5 explainers execute cleanly.

In [ ]:
smoke_model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "Qwen/Qwen3-8B-Base", "Qwen/Qwen2.5-14B", "microsoft/phi-2"]

!python scripts/run_rq2.py \
    --model {smoke_model} \
    --samples 2 \
    --operator both \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --max-evals-shap 100 \
    --lime-samples 100 \
    --ig-steps 10 \
    --output-dir scratch/smoke_test_rq2 \
    --no-resume

### 4. Main RQ2 Faithfulness Experiment (Milestone Auto-Resume: 310 -> 500)
- When `operator_mode = "both"`, each XAI explanation (`SHAP`, `LIME`, `IG`, `Occlusion`, `GradxInput`) is computed **only once per sentence** and immediately evaluated under both `deletion` and `embedding_mask` operators (cutting GPU runtime in half!).
- Checkpoints are flushed to Google Drive every 10 samples (`rq2_faithfulness_{tag}_deletion` and `rq2_faithfulness_{tag}_embedding_mask`).
- `N = 310` matches the exact 310 SST-2 validation sentences (`seed=42`) evaluated in RQ1.

In [ ]:
#@title Main RQ2 Faithfulness Parameters
model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "Qwen/Qwen3-8B-Base", "Qwen/Qwen2.5-14B", "microsoft/phi-2"]
operator_mode = "both" #@param ["both", "deletion", "embedding_mask"]
milestones = "310" #@param ["310", "310,500", "300,500", "300", "500", "872"]
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

shap_max_evals = 2500 #@param {type:"integer"}
lime_samples = 1000 #@param {type:"integer"}
ig_steps = 100 #@param {type:"integer"}
ig_baseline = "zero" #@param ["zero", "mean", "pad"]
sub_batch_size = 0 #@param {type:"integer"}

max_run_samples = 0 #@param {type:"integer"}
overwrite_methods = False #@param {type:"boolean"}
auto_shutdown = True #@param {type:"boolean"}

tag = model.split("/")[-1]
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
overwrite_flag = "--overwrite-methods" if overwrite_methods else ""
sub_batch_flag = f"--sub-batch-size {sub_batch_size}" if sub_batch_size > 0 else ""
target_list = [int(x.strip()) for x in milestones.split(",") if x.strip()]

rq1_attr_candidates = [
    os.path.join(DRIVE_RESULTS, f"rq1_stability_{tag}", "rq1_attributions.jsonl"),
    os.path.join("results", f"rq1_stability_{tag}", "rq1_attributions.jsonl"),
]
rq1_attr_path = next((p for p in rq1_attr_candidates if os.path.exists(p)), None)
rq1_attr_flag = f'--rq1-attributions "{rq1_attr_path}"' if rq1_attr_path else ""
if rq1_attr_path:
    print(f"[Cache] Detected RQ1 attributions file: {rq1_attr_path}")

if operator_mode == "both":
    base_output_dir = os.path.join(DRIVE_RESULTS, f"rq2_faithfulness_{tag}")
else:
    base_output_dir = os.path.join(DRIVE_RESULTS, f"rq2_faithfulness_{tag}_{operator_mode}")

for target_n in target_list:
    print(f"\n{'='*75}")
    print(f"  RQ2 FAITHFULNESS MILESTONE: N = {target_n} | OP: {operator_mode} | MODEL: {model}")
    print(f"  Base Output Directory: {base_output_dir}")
    print(f"{'='*75}\n")
    !python scripts/run_rq2.py \
        --model {model} \
        --samples {target_n} \
        --seed {seed} \
        --operator {operator_mode} \
        --methods {methods} \
        --max-evals-shap {shap_max_evals} \
        --lime-samples {lime_samples} \
        --ig-steps {ig_steps} \
        --ig-baseline {ig_baseline} \
        {sub_batch_flag} \
        {rq1_attr_flag} \
        {limit_flag} \
        {overwrite_flag} \
        --output-dir "{base_output_dir}"

import pandas as pd
from IPython.display import display
rq1_dir = os.path.join(DRIVE_RESULTS, f"rq1_stability_{tag}")
if not os.path.exists(os.path.join(rq1_dir, "rq1_stability_summary.csv")):
    local_rq1_dir = os.path.join("results", f"rq1_stability_{tag}")
    if os.path.exists(os.path.join(local_rq1_dir, "rq1_stability_summary.csv")):
        rq1_dir = local_rq1_dir

for op in (["deletion", "embedding_mask"] if operator_mode == "both" else [operator_mode]):
    op_dir = os.path.join(DRIVE_RESULTS, f"rq2_faithfulness_{tag}_{op}")
    summary_csv = os.path.join(op_dir, "rq2_faithfulness_summary.csv")
    stats_csv = os.path.join(op_dir, "rq2_statistical_tests.csv")
    if os.path.exists(summary_csv):
        print(f"\n=== RQ2 Faithfulness Summary ({tag} | Operator: {op}) ===")
        display(pd.read_csv(summary_csv).round(4))
        if os.path.exists(stats_csv):
            print(f"=== RQ2 Wilcoxon / Holm Tests ({tag} | Operator: {op}) ===")
            display(pd.read_csv(stats_csv).round(4))
        if os.path.exists(os.path.join(rq1_dir, "rq1_stability_summary.csv")):
            rq3_dir = os.path.join(DRIVE_RESULTS, f"rq3_joint_{tag}_{op}")
            !python src/experiments/link_rq1_rq2.py --rq1-dir "{rq1_dir}" --rq2-dir "{op_dir}" --output-dir "{rq3_dir}"

if operator_mode == "both":
    comp_csv = os.path.join(f"{base_output_dir}_operator_comparison", "operator_robustness_methods.csv")
    if os.path.exists(comp_csv):
        print(f"\n=== Cross-Operator Robustness (deletion vs embedding_mask) ===")
        display(pd.read_csv(comp_csv).round(4))

if auto_shutdown and IN_COLAB_HOSTED:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()

### 5. Results Summary Table & Faithfulness Degradation Curves

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

for op in ["deletion", "embedding_mask"]:
    op_dir = os.path.join(DRIVE_RESULTS, f"rq2_faithfulness_{tag}_{op}")
    summary_csv = os.path.join(op_dir, "rq2_faithfulness_summary.csv")
    stats_csv = os.path.join(op_dir, "rq2_statistical_tests.csv")
    if os.path.exists(summary_csv):
        print(f"\n=== RQ2 Faithfulness Summary ({tag} | Operator: {op}) ===")
        df_sum = pd.read_csv(summary_csv)
        display(df_sum.round(4))
        if os.path.exists(stats_csv):
            display(pd.read_csv(stats_csv).round(4))
        for plot_name in [
            "rq2_comprehensiveness_curves.png",
            "rq2_morf_vs_lerf_and_sufficiency.png",
            "rq2_aopc_comparison.png",
        ]:
            plot_path = os.path.join(op_dir, "plots", plot_name)
            if os.path.exists(plot_path):
                display(Image(plot_path, width=800))
    else:
        print(f"Summary file {summary_csv} not found yet.")

comp_plot = os.path.join(DRIVE_RESULTS, f"rq2_faithfulness_{tag}_operator_comparison", "plots", "operator_robustness_comparison.png")
if os.path.exists(comp_plot):
    print("\n=== Cross-Operator Robustness Comparison ===")
    display(Image(comp_plot, width=850))

### 6. Manual Session Teardown (Stop Compute Units Billing)

In [ ]:
if IN_COLAB_HOSTED:
    from google.colab import runtime
    print("Disconnecting Colab runtime...")
    runtime.unassign()